# CSV vs BBox Annotations

This notebook checks whether each unique `image` value in `results/vggnet16/ALL_vggnet.csv` has a corresponding ImageNet bounding-box annotation archive in `analysis/bboxes_annotations`.


In [1]:
from __future__ import annotations

import csv
import tarfile
from pathlib import Path

try:
    import pandas as pd
except ImportError:
    pd = None

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'results/vggnet16/ALL_vggnet.csv').exists() and (candidate / 'analysis/bboxes_annotations').exists():
            return candidate
    raise FileNotFoundError('Could not locate the XAIV repository root from the current working directory.')


ROOT = find_repo_root()
CSV_PATH = ROOT / 'results/vggnet16/ALL_vggnet.csv'
ANN_DIR = ROOT / 'analysis/bboxes_annotations'
REPORT_PATH = ROOT / 'analysis/bbox_image_archive_matches.csv'

print(f'Repository root: {ROOT}')
print(f'CSV path: {CSV_PATH}')
print(f'Annotation directory: {ANN_DIR}')


Repository root: /Users/zd3504phd/Desktop/XAIV
CSV path: /Users/zd3504phd/Desktop/XAIV/results/vggnet16/ALL_vggnet.csv
Annotation directory: /Users/zd3504phd/Desktop/XAIV/analysis/bboxes_annotations


In [2]:
with CSV_PATH.open(newline='') as f:
    rows = list(csv.DictReader(f))

unique_images = sorted({row['image'] for row in rows if row.get('image')})

records = []
for image_name in unique_images:
    synset = image_name.split('_', 1)[0]
    archive_path = ANN_DIR / f'{synset}.tar.gz'
    archive_exists = archive_path.exists()
    xml_count = 0
    sample_xmls = []

    if archive_exists:
        with tarfile.open(archive_path, 'r:gz') as tf:
            xml_members = [name for name in tf.getnames() if name.endswith('.xml')]
        xml_count = len(xml_members)
        sample_xmls = xml_members[:3]

    records.append({
        'image': image_name,
        'synset': synset,
        'archive_exists': archive_exists,
        'archive_path': str(archive_path),
        'xml_count': xml_count,
        'sample_annotations': ' | '.join(sample_xmls),
    })

missing_records = [record for record in records if not record['archive_exists']]
summary = {
    'total_csv_rows': len(rows),
    'unique_csv_images': len(unique_images),
    'matched_synset_archives': len(records) - len(missing_records),
    'missing_synset_archives': len(missing_records),
}

fieldnames = ['image', 'synset', 'archive_exists', 'archive_path', 'xml_count', 'sample_annotations']
with REPORT_PATH.open('w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(records)

if pd is not None:
    summary_df = pd.DataFrame([summary])
    match_df = pd.DataFrame(records).sort_values(['archive_exists', 'image'], ascending=[False, True])
    display(summary_df)
    display(match_df.head(10))
else:
    print(summary)
    print(records[:10])

print(f'Match report saved to: {REPORT_PATH}')
print(f'Missing matches: {len(missing_records)}')


,total_csv_rows,unique_csv_images,matched_synset_archives,missing_synset_archives
0,3261,182,0,182


,image,synset,archive_exists,archive_path,xml_count,sample_annotations
0,n01443537_goldfish,n01443537,False,/Users/zd3504phd/Desktop/XAIV/analysis/bboxes_...,0,
1,n01484850_great_white_shark,n01484850,False,/Users/zd3504phd/Desktop/XAIV/analysis/bboxes_...,0,
2,n01491361_tiger_shark,n01491361,False,/Users/zd3504phd/Desktop/XAIV/analysis/bboxes_...,0,
3,n01496331_electric_ray,n01496331,False,/Users/zd3504phd/Desktop/XAIV/analysis/bboxes_...,0,
4,n01514668_cock,n01514668,False,/Users/zd3504phd/Desktop/XAIV/analysis/bboxes_...,0,
5,n01514859_hen,n01514859,False,/Users/zd3504phd/Desktop/XAIV/analysis/bboxes_...,0,
6,n01518878_ostrich,n01518878,False,/Users/zd3504phd/Desktop/XAIV/analysis/bboxes_...,0,
7,n01530575_brambling,n01530575,False,/Users/zd3504phd/Desktop/XAIV/analysis/bboxes_...,0,
8,n01531178_goldfinch,n01531178,False,/Users/zd3504phd/Desktop/XAIV/analysis/bboxes_...,0,
9,n01532829_house_finch,n01532829,False,/Users/zd3504phd/Desktop/XAIV/analysis/bboxes_...,0,


Match report saved to: /Users/zd3504phd/Desktop/XAIV/analysis/bbox_image_archive_matches.csv
Missing matches: 182


In [3]:
def search_image(query: str):
    exact_matches = [record for record in records if record['image'] == query]
    if exact_matches:
        matches = exact_matches
    else:
        lowered = query.lower()
        matches = [record for record in records if lowered in record['image'].lower()]

    if pd is not None:
        result_df = pd.DataFrame(matches, columns=records[0].keys() if records else None)
        display(result_df)
    else:
        print(matches)

    return matches


search_image('goldfish')


,image,synset,archive_exists,archive_path,xml_count,sample_annotations
0,n01443537_goldfish,n01443537,False,/Users/zd3504phd/Desktop/XAIV/analysis/bboxes_...,0,


[{'image': 'n01443537_goldfish',
  'synset': 'n01443537',
  'archive_exists': False,
  'archive_path': '/Users/zd3504phd/Desktop/XAIV/analysis/bboxes_annotations/n01443537.tar.gz',
  'xml_count': 0,
  'sample_annotations': ''}]